En este proyecto, he configurado un asistente virtual utilizando la API de HuggingFace. El objetivo es que el modelo mantenga una conversación fluida y coherente.

**System Prompt definido:** "Eres un tutor de programación amable y paciente. Tu misión es ayudar al usuario a entender conceptos de Python de manera sencilla, utilizando analogías cuando sea necesario y respondiendo siempre en español."

**Justificación:** Elegí este rol de 'tutor' porque me permite probar si el modelo realmente puede simplificar temas complejos. Configuré el tono para que sea amable, ya que esto mejora la experiencia del usuario final.

In [14]:
# ── Instalar la librería de HuggingFace ──────────────────
!pip install -q requests
import requests
import json
print("✓ Listo para conectar a HuggingFace")

✓ Listo para conectar a HuggingFace


In [42]:
import requests
import json

# ── TU TOKEN de HuggingFace ──────────────────────────────
# Ve a huggingface.co → Settings → Access Tokens → New Token
HF_TOKEN = "aqui" # ¡Asegúrate de que este es tu token REAL y con permisos 'read'!

# ── Modelo a usar (gratuito) ─────────────────────────────
MODELO = "HuggingFaceH4/zephyr-7b-beta"

# Reemplaza tu línea de API_URL por esta:

API_URL = "https://router.huggingface.co/models/HuggingFaceH4/zephyr-7b-beta"
HEADERS = {"Authorization": f"Bearer {HF_TOKEN}"}

print(f"✓ Modelo configurado: {MODELO}")
print(f"✓ URL de la API: {API_URL}")

✓ Modelo configurado: HuggingFaceH4/zephyr-7b-beta
✓ URL de la API: https://router.huggingface.co/models/HuggingFaceH4/zephyr-7b-beta


Una vez instaladas las librerías, conecto mi código con el modelo Zephyr-7b-beta. Este modelo es ideal para esta tarea porque está optimizado para seguir instrucciones en formato de chat. He configurado los HEADERS para autenticarme con mi token personal y asegurar que la conexión sea privada.

In [43]:
# ── Función para llamar al modelo con historial ────────────────────────
def llamar_modelo(mensaje, historial, temperatura=0.7, max_tokens=200):
    # Formateamos el mensaje para que el modelo entienda el contexto
    prompt = f"System: Eres un tutor de programación amable y paciente. Tu misión es ayudar al usuario a entender conceptos de Python de manera sencilla, utilizando analogías cuando sea necesario y respondiendo siempre en español.\n__"
    for user_msg, bot_msg in historial:
        prompt += f"User: {user_msg}\nAssistant: {bot_msg}\n"
    prompt += f"User: {mensaje}\nAssistant:"

    payload = {
        "inputs": prompt,
        "parameters": {
            "temperature": temperatura,
            "max_new_tokens": max_tokens,
            "return_full_text": False
        }
    }

    respuesta = requests.post(API_URL, headers=HEADERS, json=payload)

    if respuesta.status_code == 200:
        return respuesta.json()[0]["generated_text"].split("User:")[0].strip()
    else:
        return f"Error {respuesta.status_code}: {respuesta.text}"

# ── Milestone 4: Bucle de conversación interactivo ──────────────────────
historial_chat = []
print("🤖 Chatbot iniciado. Escribe 'salir' para terminar.")

# Realizaremos los 5 turnos de conversación aquí
for i in range(5):
    usuario = input(f"Turno {i+1} - Tú: ")
    if usuario.lower() == "salir":
        break

    respuesta_bot = llamar_modelo(usuario, historial_chat)
    print(f"Chatbot: {respuesta_bot}\n")

    # Guardamos en la memoria
    historial_chat.append((usuario, respuesta_bot))

print("✓ Conversación finalizada.")

🤖 Chatbot iniciado. Escribe 'salir' para terminar.
Turno 1 - Tú: salir
✓ Conversación finalizada.


In [44]:
# ── Bucle de conversación ───────────────────────────────
print("Chatbot activo (escribe 'salir' para terminar)")
while True:
    usuario = input("Tú: ")
    if usuario.lower() == "salir":
        break
    respuesta = llamar_modelo(usuario, historial_chat)
    print(f"Chatbot: {respuesta}")

Chatbot activo (escribe 'salir' para terminar)
Tú: salir


Análisis de Datos de la Sesión (Bonus)
Finalmente, implementé un contador de palabras para monitorizar el volumen de información generado. Esto me ayuda a entender cuántos recursos (tokens aproximados) se consumieron durante la interacción.

In [5]:
# ── Milestone 5: Estadísticas ──────────────────────────
total_palabras = sum(len(u.split()) + len(b.split()) for u, b in historial_chat)
print(f"📊 Estadísticas de la sesión:")
print(f"- Turnos completados: {len(historial_chat)}")
print(f"- Total de palabras procesadas: {total_palabras}")

📊 Estadísticas de la sesión:
- Turnos completados: 0
- Total de palabras procesadas: 0


Durante el desarrollo de este chatbot, enfrenté varias limitaciones técnicas significativas. La más notable fue la latencia de la API de HuggingFace; el modelo tardaba en "despertar", provocando errores de tiempo de espera (503). También noté que la ventana de contexto es limitada; en conversaciones largas, el modelo ignora las primeras instrucciones. Otra dificultad fue el manejo de formatos: sin una limpieza adecuada, el modelo genera texto basura o imita al usuario. Esta práctica me permitió entender que el éxito de la IA depende tanto del código como de un "prompt engineering" estratégico y bien estructurado para guiar las respuestas.